In [1]:
from pathlib import Path
import copy
import sys
import time
import numpy as np

PROJECT_ROOT = Path('/home/baiyu/LearnStageConstraints')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from runners.run_benchmark import _deep_merge, _load_env_config, _load_method_config
from experiments.config_loader import resolve_dataset_method_override
from experiments.unified_experiment import run_experiment

config_root = PROJECT_ROOT / 'configs'
dataset_cfg = _load_env_config(config_root, 'BarClean')
dataset_method_overrides = dict(dataset_cfg.pop('method_overrides', {}))
method_cfg = _load_method_config(config_root, 'map_balanced_pooled')
method_cfg = _deep_merge(method_cfg, resolve_dataset_method_override('map_balanced_pooled', dataset_method_overrides))
method_cfg['seed'] = 0
method_cfg['disable_plots'] = True
method_cfg['verbose'] = False
method_cfg['plot_dir'] = str(PROJECT_ROOT / 'outputs/diagnostics/barclean_obs_eq_prior_threshold/notebook')

print({'python': sys.executable, 'source_demo_ids': dataset_cfg.get('source_demo_ids'), 'max_iter': method_cfg.get('max_iter'), 'active_prior_obs': method_cfg['map_activation_prior'][0], 'active_mode_prior_obs': {k: v[0] for k, v in method_cfg['map_active_mode_prior'].items()}})

{'python': '/home/baiyu/miniforge3/envs/segment/bin/python', 'source_demo_ids': [3, 5, 6, 7, 8, 9], 'max_iter': 8, 'active_prior_obs': 0.5, 'active_mode_prior_obs': {'eq': 0.3333333333333333, 'lb': 0.3333333333333333, 'ub': 0.3333333333333333}}


In [2]:
started = time.perf_counter()
baseline_result = run_experiment(dataset_name='BarClean', method_name='map_balanced_pooled', dataset_kwargs=dataset_cfg, method_kwargs=method_cfg)
baseline_joint = baseline_result['joint_result']
baseline_model = baseline_joint['model']
print('elapsed_s', round(time.perf_counter() - started, 2))
print('mode-related attrs', [name for name in dir(baseline_model) if 'mode' in name.lower() and name.endswith('_')])
print('learned semantics', baseline_joint['metrics']['ConstraintLearnedSemanticsMatrix'])
print('cutpoints', baseline_joint['cutpoints_hat'])

elapsed_s 157.75
mode-related attrs ['map_shared_mode_costs_', 'map_shared_mode_votes_']
learned semantics [['lower_bound', '', '', '', '', '', ''], ['', 'target_value', 'target_value', '', 'target_value', 'target_value', 'target_value'], ['', '', '', '', '', '', ''], ['target_value', 'target_value', '', 'target_value', 'target_value', 'target_value', 'target_value'], ['', '', '', '', '', '', '']]
cutpoints [[33, 62, 83, 109], [35, 69, 94, 115], [35, 69, 115, 138], [26, 53, 82, 104], [28, 56, 84, 103], [27, 56, 80, 106]]


In [3]:
print('shared costs shape/type', type(baseline_model.map_shared_mode_costs_))
print('S4 obs shared costs', baseline_model.map_shared_mode_costs_[3][0])
print('S4 obs vote diagnostic', baseline_model.map_shared_mode_votes_[3][0])
print('candidate state attrs', [name for name in baseline_model.__dict__ if any(token in name.lower() for token in ('selected', 'stage_end', 'interval', 'info'))])

shared costs shape/type <class 'list'>
S4 obs shared costs {'inactive': -0.9766507550965003, 'eq': -9.028623782292996, 'lb': -2.7262264166064134, 'ub': -2.4846374108318634}
S4 obs vote diagnostic {'aggregation': 'demo_balanced_pooled', 'selected_mode': 'eq', 'majority_required': None, 'vote_counts': {}, 'demo_votes': [], 'demo_mean_nlls': [], 'demo_vote_scores': [], 'refit_enabled': False, 'pre_refit_vector': None, 'post_refit_vector': None, 'refit_demo_indices': [], 'aggregate_mode': 'eq', 'pooled_mode': 'eq', 'pooled_costs': {'inactive': -0.9766507550965003, 'eq': -9.028623782292996, 'lb': -2.7262264166064134, 'ub': -2.4846374108318634}}
candidate state attrs ['selected_raw_feature_ids', 'selected_feature_columns', 'stage_ends_', '_map_interval_stats_cache']


In [4]:
import math

base_costs = baseline_model.map_shared_mode_costs_[3][0]
prior0 = {'inactive': -math.log(0.5), 'eq': -math.log(0.5 / 3), 'lb': -math.log(0.5 / 3), 'ub': -math.log(0.5 / 3)}
likelihood = {mode: base_costs[mode] - prior0[mode] for mode in base_costs}
ratio = math.exp(likelihood['eq'] - likelihood['lb'])
q_eq_lb = ratio / (2.0 + ratio)
q_eq_inactive = math.exp(likelihood['eq'] - likelihood['inactive'])
small_q_lb_cost = likelihood['lb'] - math.log(0.25 * (1.0 - q_eq_inactive))
inactive_cost = likelihood['inactive'] - math.log(0.5)
print({'likelihood_costs': likelihood, 'eq_to_lb_threshold': q_eq_lb, 'eq_to_inactive_crossing': q_eq_inactive, 'lb_cost_at_eq_inactive_crossing': small_q_lb_cost, 'inactive_cost': inactive_cost})

def run_obs_eq_prior(p_eq):
    cfg = copy.deepcopy(method_cfg)
    mode_prior = copy.deepcopy(cfg['map_active_mode_prior'])
    mode_prior['eq'][0] = float(p_eq)
    mode_prior['lb'][0] = (1.0 - float(p_eq)) / 2.0
    mode_prior['ub'][0] = (1.0 - float(p_eq)) / 2.0
    cfg['map_active_mode_prior'] = mode_prior
    started = time.perf_counter()
    result = run_experiment(dataset_name='BarClean', method_name='map_balanced_pooled', dataset_kwargs=dataset_cfg, method_kwargs=cfg)
    joint = result['joint_result']
    model = joint['model']
    return {'p_eq': p_eq, 'elapsed_s': time.perf_counter() - started, 's4_obs_semantics': joint['metrics']['ConstraintLearnedSemanticsMatrix'][3][0], 's4_obs_costs': model.map_shared_mode_costs_[3][0], 'cutpoints': joint['cutpoints_hat'], 'semantics': joint['metrics']['ConstraintLearnedSemanticsMatrix'], 'result': result}

{'likelihood_costs': {'inactive': -1.6697979356564456, 'eq': -10.82038325152105, 'lb': -4.517985885834468, 'ub': -4.276396880059918}, 'eq_to_lb_threshold': 0.0009151156741027767, 'eq_to_inactive_crossing': 0.00010615764880239528, 'lb_cost_at_eq_inactive_crossing': -3.1315853614306532, 'inactive_cost': -0.9766507550965003}


In [5]:
run_1e3 = run_obs_eq_prior(1e-3)
print({key: value for key, value in run_1e3.items() if key != 'result' and key != 'semantics' and key != 'cutpoints'})
print('cutpoints', run_1e3['cutpoints'])
print('semantics', run_1e3['semantics'])

KeyboardInterrupt: 

In [6]:
run_zero = run_obs_eq_prior(0.0)
print({'p_eq': run_zero['p_eq'], 'elapsed_s': run_zero['elapsed_s'], 's4_obs_semantics': run_zero['s4_obs_semantics'], 's4_obs_costs': run_zero['s4_obs_costs']})
print('cutpoints', run_zero['cutpoints'])
print('semantics', run_zero['semantics'])

{'p_eq': 0.0, 'elapsed_s': 127.15640387358144, 's4_obs_semantics': 'lower_bound', 's4_obs_costs': {'inactive': -0.9547604605788648, 'eq': inf, 'lb': -3.08423845374626, 'ub': -2.5769500729607397}}
cutpoints [[33, 62, 83, 108], [35, 69, 92, 116], [35, 69, 115, 138], [25, 53, 82, 104], [28, 56, 84, 102], [27, 56, 80, 106]]
semantics [['lower_bound', '', '', '', '', '', ''], ['', 'target_value', 'target_value', '', 'target_value', 'target_value', 'target_value'], ['', '', '', '', '', '', ''], ['lower_bound', 'target_value', '', 'target_value', 'target_value', 'target_value', 'target_value'], ['', '', '', '', '', '', '']]


In [7]:
from experiments.artifacts import _extract_learned_constraint_artifact

zero_result = run_zero['result']
zero_joint = zero_result['joint_result']
zero_metrics = zero_joint['metrics']
baseline_metrics = baseline_joint['metrics']
feature_names = list(baseline_metrics['ConstraintFeatureNames'])
baseline_semantics = np.asarray(baseline_metrics['ConstraintLearnedSemanticsMatrix'], dtype=object)
zero_semantics = np.asarray(zero_metrics['ConstraintLearnedSemanticsMatrix'], dtype=object)
baseline_values = np.asarray(baseline_metrics['ConstraintLearnedValueMatrix'], dtype=float)
zero_values = np.asarray(zero_metrics['ConstraintLearnedValueMatrix'], dtype=float)
feature_scales = np.asarray(baseline_metrics['ConstraintFeatureScales'], dtype=float)

mode_changes = []
value_changes = []
for stage in range(baseline_semantics.shape[0]):
    for feature, name in enumerate(feature_names):
        old_mode = str(baseline_semantics[stage, feature]) or 'inactive'
        new_mode = str(zero_semantics[stage, feature]) or 'inactive'
        if old_mode != new_mode:
            mode_changes.append({'stage': stage + 1, 'feature': name, 'uniform': old_mode, 'eq0': new_mode, 'uniform_value': None if not np.isfinite(baseline_values[stage, feature]) else float(baseline_values[stage, feature]), 'eq0_value': None if not np.isfinite(zero_values[stage, feature]) else float(zero_values[stage, feature])})
        if old_mode != 'inactive' and new_mode != 'inactive' and np.isfinite(baseline_values[stage, feature]) and np.isfinite(zero_values[stage, feature]):
            delta = float(zero_values[stage, feature] - baseline_values[stage, feature])
            value_changes.append({'stage': stage + 1, 'feature': name, 'uniform_mode': old_mode, 'eq0_mode': new_mode, 'uniform_value': float(baseline_values[stage, feature]), 'eq0_value': float(zero_values[stage, feature]), 'delta': delta, 'abs_delta_over_feature_scale': abs(delta) / float(feature_scales[feature])})

baseline_cuts = np.asarray(baseline_joint['cutpoints_hat'], dtype=int)
zero_cuts = np.asarray(zero_joint['cutpoints_hat'], dtype=int)
cut_delta = zero_cuts - baseline_cuts
changed_labels = []
for demo, (old_cuts, new_cuts) in enumerate(zip(baseline_cuts, zero_cuts)):
    length = len(baseline_result['dataset'].demos[demo])
    old_labels = np.searchsorted(old_cuts, np.arange(length), side='left')
    new_labels = np.searchsorted(new_cuts, np.arange(length), side='left')
    changed_labels.append(int(np.sum(old_labels != new_labels)))

baseline_artifact = _extract_learned_constraint_artifact(dataset_name='BarClean', method_name='map_balanced_pooled', method_seed=0, result=baseline_result)
zero_artifact = _extract_learned_constraint_artifact(dataset_name='BarClean', method_name='map_balanced_pooled', method_seed=0, result=zero_result)
baseline_endpoints = np.asarray(baseline_artifact['stage_endpoint_poses_bar'], dtype=float)
zero_endpoints = np.asarray(zero_artifact['stage_endpoint_poses_bar'], dtype=float)
endpoint_position_delta_mm = 1000.0 * np.linalg.norm(zero_endpoints[:, :3] - baseline_endpoints[:, :3], axis=1)
endpoint_angle_delta_deg = []
for old_q, new_q in zip(baseline_endpoints[:, 3:], zero_endpoints[:, 3:]):
    dot = abs(float(np.dot(old_q / np.linalg.norm(old_q), new_q / np.linalg.norm(new_q))))
    endpoint_angle_delta_deg.append(float(np.degrees(2.0 * np.arccos(np.clip(dot, -1.0, 1.0)))))

print('MODE CHANGES', mode_changes)
print('CUTPOINT DELTA eq0-uniform', cut_delta.tolist())
print('CUTPOINT SUMMARY', {'changed_of_24': int(np.sum(cut_delta != 0)), 'mean_abs_samples': float(np.mean(np.abs(cut_delta))), 'max_abs_samples': int(np.max(np.abs(cut_delta))), 'changed_stage_labels_per_demo': changed_labels, 'changed_stage_labels_total': int(sum(changed_labels))})
print('ENDPOINT POSITION DELTA mm', endpoint_position_delta_mm.tolist())
print('ENDPOINT ORIENTATION DELTA deg', endpoint_angle_delta_deg)
print('METRICS', {key: {'uniform': baseline_metrics.get(key), 'eq0': zero_metrics.get(key)} for key in ('MeanAbsCutpointError', 'SemanticConstraintPrecision', 'SemanticConstraintRecall', 'SemanticConstraintF1', 'PredictedConstraintCount')})
print('VALUE CHANGES')
for row in value_changes:
    print(row)

MODE CHANGES [{'stage': 4, 'feature': 'obs_dist', 'uniform': 'target_value', 'eq0': 'lower_bound', 'uniform_value': 0.3972037160599975, 'eq0_value': 0.3854802355375947}]
CUTPOINT DELTA eq0-uniform [[0, 0, 0, -1], [0, 0, -2, 1], [0, 0, 0, 0], [-1, 0, 0, 0], [0, 0, 0, -1], [0, 0, 0, 0]]
CUTPOINT SUMMARY {'changed_of_24': 5, 'mean_abs_samples': 0.25, 'max_abs_samples': 2, 'changed_stage_labels_per_demo': [1, 3, 0, 1, 1, 0], 'changed_stage_labels_total': 6}
ENDPOINT POSITION DELTA mm [1.2141920042158314, 0.0, 3.7194699013759016, 2.4976693475470726]
ENDPOINT ORIENTATION DELTA deg [1.3063295182694208, 0.0, 1.0468106546069862, 0.44220737323996134]
METRICS {'MeanAbsCutpointError': {'uniform': 3.375, 'eq0': 3.4583333333333335}, 'SemanticConstraintPrecision': {'uniform': 0.9166666666666666, 'eq0': 0.9166666666666666}, 'SemanticConstraintRecall': {'uniform': 1.0, 'eq0': 1.0}, 'SemanticConstraintF1': {'uniform': 0.9565217391304348, 'eq0': 0.9565217391304348}, 'PredictedConstraintCount': {'uniform'

In [8]:
from pathlib import Path
import copy
import json
import math
import numpy as np
from rosbags.highlevel import AnyReader

PLANNER_SRC = PROJECT_ROOT / 'robot/stage_cons_iiwa14/ros_ws/src/stage_constraint_planner/src'
if str(PLANNER_SRC) not in sys.path:
    sys.path.insert(0, str(PLANNER_SRC))

from stage_constraint_planner.constraint_artifact import _task_fixed_term_parameters
from stage_constraint_planner.optimizer import StageConstraintTrajectoryOptimizer, transform_pose

bag_path = PROJECT_ROOT / 'robot/stage_cons_iiwa14/data/demos/BarClean/20260829T153716_200912Z_real_task/real_task.bag'
scene_topics = {
    '/vrpn_client_node/baiyu_bar/pose_from_iiwa14': 'bar',
    '/vrpn_client_node/baiyu_obs_bar/pose_from_iiwa14': 'obstacle',
}
samples = {name: [] for name in scene_topics.values()}
plan_timestamp = None
with AnyReader([bag_path]) as reader:
    connections = [connection for connection in reader.connections if connection.topic in set(scene_topics) | {'/stage_cons/plan'}]
    for connection, timestamp, rawdata in reader.messages(connections=connections):
        if connection.topic == '/stage_cons/plan' and plan_timestamp is None:
            plan_timestamp = timestamp
            continue
        name = scene_topics.get(connection.topic)
        if name is None:
            continue
        message = reader.deserialize(rawdata, connection.msgtype)
        pose = message.pose
        samples[name].append((timestamp, np.asarray([
            pose.position.x, pose.position.y, pose.position.z,
            pose.orientation.x, pose.orientation.y, pose.orientation.z, pose.orientation.w,
        ], dtype=float)))

def nearest_scene_pose(name):
    return min(samples[name], key=lambda item: abs(item[0] - plan_timestamp))[1]

scene_rotation = np.asarray([[0.0, 0.0, 1.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0]])
bar_pose = transform_pose(nearest_scene_pose('bar'), scene_rotation, np.zeros(3))
obstacle_pose = transform_pose(nearest_scene_pose('obstacle'), scene_rotation, np.zeros(3))
obstacle = {'type': 'circle', 'center': obstacle_pose[:3], 'radius': 0.025}

gui_settings = json.loads((PROJECT_ROOT / 'robot/stage_cons_iiwa14/data/gui_settings.json').read_text())
start_dict = gui_settings['start_by_task']['BarClean']
goal_dict = gui_settings['goal_by_task']['BarClean']
def gui_pose(values):
    return np.asarray([values['x'], values['y'], values['z'], values['qx'], values['qy'], values['qz'], values['qw']], dtype=float)
start_pose = gui_pose(start_dict)
goal_pose = gui_pose(goal_dict)

base_config_path = PROJECT_ROOT / 'robot/stage_cons_iiwa14/ros_ws/src/stage_constraint_planner/config/bar_clean_true.json'
base_config = json.loads(base_config_path.read_text())
current_eq_value = float(next(pair['value'] for pair in baseline_artifact['feature_stage_modes'] if int(pair['stage']) == 3 and pair['feature_name'] == 'obs_dist'))
learned_lb_value = float(next(pair['value'] for pair in zero_artifact['feature_stage_modes'] if int(pair['stage']) == 3 and pair['feature_name'] == 'obs_dist'))

def planning_config(obs_mode):
    config = copy.deepcopy(base_config)
    endpoint_poses = copy.deepcopy(baseline_artifact['stage_endpoint_poses_bar'])
    config['stage_endpoint_poses_bar'] = endpoint_poses
    config['stage_endpoint_positions_bar'] = [pose[:3] for pose in endpoint_poses]
    true_terms = config['constraint_terms']
    terms = []
    for pair in baseline_artifact['feature_stage_modes']:
        stage = int(pair['stage'])
        feature = str(pair['feature_name'])
        semantics = str(pair['mode'])
        value = pair.get('value')
        if stage == 3 and feature == 'obs_dist':
            if obs_mode == 'inactive':
                continue
            semantics = 'lower_bound' if obs_mode == 'lower_bound' else 'target_value'
            value = learned_lb_value if obs_mode == 'lower_bound' else current_eq_value
        if semantics == 'inactive':
            continue
        scale, weight = _task_fixed_term_parameters(true_terms, stage, feature)
        terms.append({'feature_name': feature, 'stage': stage, 'semantics': semantics, 'value': float(value), 'scale': scale, 'weight': weight})
    config['constraint_terms'] = terms
    return config

def make_optimizer(config):
    planner = config['planner']
    return StageConstraintTrajectoryOptimizer(
        config,
        control_spacing=float(planner['control_spacing_m']),
        output_spacing=float(planner['output_spacing_m']),
        output_axis_spacing=math.radians(float(planner['output_axis_spacing_deg'])),
        min_control_points=int(planner['min_control_points']),
        max_control_points=int(planner['max_control_points']),
        max_nfev=int(planner['max_nfev']),
        multi_start=int(planner['multi_start']),
    )

planned_modes = {}
for obs_mode in ('inactive', 'lower_bound', 'eq'):
    started = time.perf_counter()
    config = planning_config(obs_mode)
    planned = make_optimizer(config).plan(
        start_pose, goal_pose, bar_pose, obstacle,
        bar_lateral_centerline={'type': 'straight'}, seed=2026,
    )
    planned_modes[obs_mode] = planned
    print(obs_mode, {'elapsed_s': round(time.perf_counter() - started, 2), 'solver_success': planned['solver_success'], 'objective': planned['objective'], 'n_points': len(planned['positions'])})

print('scene', {'bag': str(bag_path), 'start': start_pose.tolist(), 'goal': goal_pose.tolist(), 'bar_pose': bar_pose.tolist(), 'obstacle_center': obstacle['center'].tolist(), 'eq_value': current_eq_value, 'lower_bound_value': learned_lb_value})

inactive {'elapsed_s': 5.63, 'solver_success': True, 'objective': 6.216052387265967, 'n_points': 255}
lower_bound {'elapsed_s': 9.38, 'solver_success': True, 'objective': 6.216052394318426, 'n_points': 255}
eq {'elapsed_s': 12.42, 'solver_success': True, 'objective': 10.353347996305601, 'n_points': 251}
scene {'bag': '/home/baiyu/LearnStageConstraints/robot/stage_cons_iiwa14/data/demos/BarClean/20260829T153716_200912Z_real_task/real_task.bag', 'start': [0.464, 0.3037, 0.3258, 6.123233995736766e-17, 1.0, 6.123233995736766e-17, 3.749399456654644e-33], 'goal': [0.5705, -0.0426, 0.2658, 6.123233995736766e-17, 1.0, 6.123233995736766e-17, 3.749399456654644e-33], 'bar_pose': [0.64223939, -0.08751724, 0.12537, 3.69440873971151e-17, -7.916590156524665e-17, -0.7535633892235077, 0.6573752493226226], 'obstacle_center': [0.6482, 0.1373, 0.13437], 'eq_value': 0.3972037160599975, 'lower_bound_value': 0.3854802355375947}


In [9]:
def path_length(positions):
    return float(np.sum(np.linalg.norm(np.diff(np.asarray(positions), axis=0), axis=1)))

def stage_indices(plan, stage):
    return np.flatnonzero(np.asarray(plan['stage_labels']) == stage)

def resample_scalar_by_stage(plan, values, stage, count=201):
    indices = stage_indices(plan, stage)
    positions = np.asarray(plan['positions'])[indices]
    values = np.asarray(values)[indices]
    distance = np.concatenate(([0.0], np.cumsum(np.linalg.norm(np.diff(positions, axis=0), axis=1))))
    targets = np.linspace(0.0, distance[-1], count)
    if distance[-1] <= 1e-12:
        return np.repeat(values[:1], count, axis=0)
    if values.ndim == 1:
        return np.interp(targets, distance, values)
    return np.column_stack([np.interp(targets, distance, values[:, column]) for column in range(values.shape[1])])

def aligned_difference(first, second, stages=range(5), count=201):
    position_errors = []
    axis_errors = []
    yaw_errors = []
    stage_position = {}
    for stage in stages:
        p1 = resample_scalar_by_stage(first, first['positions'], stage, count)
        p2 = resample_scalar_by_stage(second, second['positions'], stage, count)
        a1 = resample_scalar_by_stage(first, first['tool_axes'], stage, count)
        a2 = resample_scalar_by_stage(second, second['tool_axes'], stage, count)
        a1 /= np.linalg.norm(a1, axis=1, keepdims=True)
        a2 /= np.linalg.norm(a2, axis=1, keepdims=True)
        y1 = resample_scalar_by_stage(first, np.unwrap(first['tool_yaws']), stage, count)
        y2 = resample_scalar_by_stage(second, np.unwrap(second['tool_yaws']), stage, count)
        p_error = np.linalg.norm(p1 - p2, axis=1)
        a_error = np.degrees(np.arccos(np.clip(np.sum(a1 * a2, axis=1), -1.0, 1.0)))
        y_error = np.degrees(np.abs(np.arctan2(np.sin(y1 - y2), np.cos(y1 - y2))))
        position_errors.extend(p_error)
        axis_errors.extend(a_error)
        yaw_errors.extend(y_error)
        stage_position[stage] = p_error
    return {
        'position_rms_mm': 1000.0 * float(np.sqrt(np.mean(np.square(position_errors)))),
        'position_max_mm': 1000.0 * float(np.max(position_errors)),
        'axis_rms_deg': float(np.sqrt(np.mean(np.square(axis_errors)))),
        'axis_max_deg': float(np.max(axis_errors)),
        'yaw_rms_deg': float(np.sqrt(np.mean(np.square(yaw_errors)))),
        'yaw_max_deg': float(np.max(yaw_errors)),
        's4_position_rms_mm': 1000.0 * float(np.sqrt(np.mean(np.square(stage_position[3])))),
        's4_position_max_mm': 1000.0 * float(np.max(stage_position[3])),
    }

mode_summary = {}
for mode, plan in planned_modes.items():
    s4 = stage_indices(plan, 3)
    obs = np.asarray(plan['features']['obs_dist'])[s4]
    core = s4[np.asarray(plan['stage_constraint_weights'])[s4, 3] >= 0.5]
    core_obs = np.asarray(plan['features']['obs_dist'])[core]
    mode_summary[mode] = {
        'solver_success': plan['solver_success'],
        'objective': float(plan['objective']),
        'constraint_objective': float(plan['constraint_objective']),
        'n_points': len(plan['positions']),
        'boundaries': np.asarray(plan['stage_boundaries']).tolist(),
        'path_length_m': path_length(plan['positions']),
        's4_length_m': path_length(np.asarray(plan['positions'])[s4]),
        's4_obs_min_m': float(np.min(obs)),
        's4_obs_mean_m': float(np.mean(obs)),
        's4_obs_max_m': float(np.max(obs)),
        's4_core_obs_min_m': float(np.min(core_obs)),
        's4_core_obs_mean_m': float(np.mean(core_obs)),
        's4_core_obs_max_m': float(np.max(core_obs)),
        'eq_mean_abs_error_m': float(np.mean(np.abs(core_obs - current_eq_value))),
        'eq_max_abs_error_m': float(np.max(np.abs(core_obs - current_eq_value))),
        'lb_mean_violation_m': float(np.mean(np.maximum(learned_lb_value - core_obs, 0.0))),
        'lb_max_violation_m': float(np.max(np.maximum(learned_lb_value - core_obs, 0.0))),
    }

pairwise = {}
for mode in ('lower_bound', 'eq'):
    comparison = aligned_difference(planned_modes['inactive'], planned_modes[mode])
    comparison['path_length_delta_mm'] = 1000.0 * (mode_summary[mode]['path_length_m'] - mode_summary['inactive']['path_length_m'])
    comparison['s4_length_delta_mm'] = 1000.0 * (mode_summary[mode]['s4_length_m'] - mode_summary['inactive']['s4_length_m'])
    pairwise[mode + '_vs_inactive'] = comparison

direct_lb = np.linalg.norm(planned_modes['lower_bound']['positions'] - planned_modes['inactive']['positions'], axis=1)
print('MODE SUMMARY')
for mode, summary in mode_summary.items():
    print(mode, summary)
print('PAIRWISE ALIGNED')
for name, comparison in pairwise.items():
    print(name, comparison)
print('LB DIRECT SAME-GRID', {'rms_mm': 1000.0 * float(np.sqrt(np.mean(np.square(direct_lb)))), 'max_mm': 1000.0 * float(np.max(direct_lb))})

MODE SUMMARY
inactive {'solver_success': True, 'objective': 6.216052387265967, 'constraint_objective': 2.0500670229533675, 'n_points': 255, 'boundaries': [75, 139, 157, 192, 254], 'path_length_m': 1.252108298400822, 's4_length_m': 0.16616007693692897, 's4_obs_min_m': 0.3927808889902699, 's4_obs_mean_m': 0.39633437472153976, 's4_obs_max_m': 0.4042468511149273, 's4_core_obs_min_m': 0.3927808889902699, 's4_core_obs_mean_m': 0.39609284681311535, 's4_core_obs_max_m': 0.4042468511149273, 'eq_mean_abs_error_m': 0.0029586607708646014, 'eq_max_abs_error_m': 0.007043135054929817, 'lb_mean_violation_m': 0.0, 'lb_max_violation_m': 0.0}
lower_bound {'solver_success': True, 'objective': 6.216052394318426, 'constraint_objective': 2.050066978887844, 'n_points': 255, 'boundaries': [75, 139, 157, 192, 254], 'path_length_m': 1.2521090226190648, 's4_length_m': 0.1661600729147806, 's4_obs_min_m': 0.39278088905662123, 's4_obs_mean_m': 0.3963343752190043, 's4_obs_max_m': 0.40424685069425564, 's4_core_obs_min

In [10]:
def stagewise_pair(first, second):
    rows = []
    for stage in range(5):
        p1 = resample_scalar_by_stage(first, first['positions'], stage, 201)
        p2 = resample_scalar_by_stage(second, second['positions'], stage, 201)
        error = 1000.0 * np.linalg.norm(p1 - p2, axis=1)
        i1 = stage_indices(first, stage)
        i2 = stage_indices(second, stage)
        rows.append({
            'stage': stage + 1,
            'position_rms_mm': float(np.sqrt(np.mean(np.square(error)))),
            'position_max_mm': float(np.max(error)),
            'length_inactive_mm': 1000.0 * path_length(np.asarray(first['positions'])[i1]),
            'length_other_mm': 1000.0 * path_length(np.asarray(second['positions'])[i2]),
        })
    return rows

print('EQ VS INACTIVE BY STAGE')
for row in stagewise_pair(planned_modes['inactive'], planned_modes['eq']):
    print(row)
print('ENDPOINT POSITION DIFFERENCE EQ-INACTIVE mm', (1000.0 * np.linalg.norm(planned_modes['eq']['stage_endpoints_world'] - planned_modes['inactive']['stage_endpoints_world'], axis=1)).tolist())
print('ENDPOINT POSITION DIFFERENCE LB-INACTIVE mm', (1000.0 * np.linalg.norm(planned_modes['lower_bound']['stage_endpoints_world'] - planned_modes['inactive']['stage_endpoints_world'], axis=1)).tolist())

EQ VS INACTIVE BY STAGE
{'stage': 1, 'position_rms_mm': 0.0008321052967671604, 'position_max_mm': 0.0011032225734821971, 'length_inactive_mm': 367.4344086909397, 'length_other_mm': 367.4342696547404}
{'stage': 2, 'position_rms_mm': 0.0015338028102121605, 'position_max_mm': 0.008178981545408467, 'length_inactive_mm': 310.72705492226095, 'length_other_mm': 310.7278922788319}
{'stage': 3, 'position_rms_mm': 1.4995503226296756, 'position_max_mm': 2.5166945948719666, 'length_inactive_mm': 79.4959600525971, 'length_other_mm': 78.70865054043223}
{'stage': 4, 'position_rms_mm': 0.8572930027860047, 'position_max_mm': 2.5374593544358612, 'length_inactive_mm': 166.16007693692896, 'length_other_mm': 167.2995176768121}
{'stage': 5, 'position_rms_mm': 9.750721952701584, 'position_max_mm': 15.37023885492448, 'length_inactive_mm': 308.747171228249, 'length_other_mm': 292.15157856149966}
ENDPOINT POSITION DIFFERENCE EQ-INACTIVE mm [0.0007050297845256903, 0.010012331782188744, 2.5374593544358612, 0.0001